In [1]:
!pip install -q yt-dlp decord librosa soundfile h5py tqdm ijson opencv-python
!apt-get install -y -q ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 9.2 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## FULL SETUP

In [3]:
# FULL SETUP
import os, json, subprocess, warnings
import numpy as np
import librosa
import cv2
from tqdm import tqdm
from decord import VideoReader, cpu

In [5]:
# ---- CONFIG ----
SEGMENT_DURATION = 2
FPS_TARGET       = 2
AUDIO_SR         = 16000
MAX_SEGMENTS     = 15000
BATCH_SIZE       = 50

BASE_DIR        = "/content/drive/MyDrive/imagination_dataset"
TMP_VIDEO_DIR   = os.path.join(BASE_DIR, "videos")
OUTPUT_DIR      = os.path.join(BASE_DIR, "segments")   # one subfolder per segment
CHECKPOINT_PATH = os.path.join(BASE_DIR, "checkpoint_v6.json")
ANNOTATION_PATH = os.path.join(BASE_DIR, "VASTannotations150k.json")

In [ ]:
#IMPORT HELPERS FROM REPO
import sys
sys.path.append("/content/drive/MyDrive/imagination_dataset/scripts")  # Colab
# sys.path.append("../scripts")  # local laptop

from download_dataset import (
    parse_time_string,
    load_checkpoint, save_checkpoint,
    download_video,
    extract_frames, extract_audio,
    save_segment, count_segments_on_disk,
    SEGMENT_DURATION, FPS_TARGET, AUDIO_SR
)

Loading annotations...
 Setup complete. 150154 annotations loaded.


In [ ]:
# these two now take the path explicitly
ckpt = load_checkpoint(CHECKPOINT_PATH)
save_checkpoint(CHECKPOINT_PATH, {"index": i + 1, "segments": total_segments})

# save_segment now takes output_dir as first arg
save_segment(OUTPUT_DIR, total_segments, frames, audio, text)

# count_segments_on_disk too
disk_count = count_segments_on_disk(OUTPUT_DIR)

Checkpoint → index=11616, segments=10098
⚠️  Mismatch: checkpoint=10098, disk=10110 → using disk count
Resuming from annotation index 11616/150154 | segments so far: 10110


  8%|▊         | 11673/150154 [08:21<315:05:03,  8.19s/it]

  checkpoint → index=11673, segments=10160


  8%|▊         | 11731/150154 [19:03<698:36:14, 18.17s/it]

  checkpoint → index=11731, segments=10210


  8%|▊         | 11787/150154 [26:43<299:24:20,  7.79s/it]

  checkpoint → index=11787, segments=10260


  8%|▊         | 11840/150154 [33:47<351:53:11,  9.16s/it]

  checkpoint → index=11840, segments=10310


  8%|▊         | 11860/150154 [37:20<216:13:17,  5.63s/it]

## Debugging

In [ ]:
for offset in range(5):
    idx = 1975 + offset
    if idx >= len(annotations): break
    ann = annotations[idx]
    print(f"\n--- Index {idx} ---")
    print(f"url: {ann.get('url', ann.get('video_url', 'MISSING'))[:80]}")
    print(f"clip_span: {ann.get('clip_span')}")
    print(f"subtitle/vast_cap: {ann.get('subtitle', ann.get('vast_cap', ''))[:50]}")


--- Index 1975 ---
url: https://www.youtube.com/watch?v=-AawZPTw8pY
clip_span: ['00:10:38.170', '00:10:50.950']
subtitle/vast_cap: So if you look at this which has been adapted, you

--- Index 1976 ---
url: https://www.youtube.com/watch?v=Z1vLZq38tYE
clip_span: ['00:07:58.130', '00:08:05.810']
subtitle/vast_cap: What do we got? Oh, baby, Superboys steel. Thank y

--- Index 1977 ---
url: https://www.youtube.com/watch?v=zU9c3IxsKvo
clip_span: ['00:13:16.649', '00:13:28.220']
subtitle/vast_cap: It is another dollar. Please Music, He is no longe

--- Index 1978 ---
url: https://www.youtube.com/watch?v=TjfFbRbRu-0
clip_span: ['00:03:54.840', '00:04:04.080']
subtitle/vast_cap: Then I might be able to look back to the theory of

--- Index 1979 ---
url: https://www.youtube.com/watch?v=7bOrJCA4Gb8
clip_span: ['00:03:26.970', '00:03:32.010']
subtitle/vast_cap: You don't wanna do this, that's too far.


In [ ]:
import subprocess, os
test_idx = 1975
test_ann = annotations[test_idx]
test_url = test_ann.get('url') or test_ann.get('video_url')
if test_url:
    test_path = "/tmp/test_video.mp4"
    result = subprocess.run(
        ["yt-dlp", "-f", "mp4", "-o", test_path, test_url],
        capture_output=True, text=True, timeout=60
    )
    print(f"Return code: {result.returncode}")
    print(f"Stdout: {result.stdout[:200]}")
    print(f"Stderr: {result.stderr[:200]}")
    if os.path.exists(test_path):
        print(f"Downloaded size: {os.path.getsize(test_path)} bytes")
        os.remove(test_path)
    else:
        print("No file downloaded")
else:
    print("No URL found")

Return code: 1
Stdout: [youtube] Extracting URL: https://www.youtube.com/watch?v=-AawZPTw8pY
[youtube] -AawZPTw8pY: Downloading webpage
[youtube] -AawZPTw8pY: Downloading initial data API JSON
[youtube] -AawZPTw8pY: Downloa
Stderr: WARNING: "-f mp4" selects the best pre-merged mp4 format which is often not what's intended.
         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality
No file downloaded


In [ ]:
!yt-dlp --version
!yt-dlp -F https://www.youtube.com/watch?v=dQw4w9WgXcQ 2>&1 | head -20

2026.03.17
[youtube] Extracting URL: https://www.youtube.com/watch?v=dQw4w9WgXcQ
[youtube] dQw4w9WgXcQ: Downloading webpage
[youtube] dQw4w9WgXcQ: Downloading initial data API JSON
[youtube] dQw4w9WgXcQ: Downloading android vr player API JSON
[info] Available formats for dQw4w9WgXcQ:
ID  EXT   RESOLUTION FPS CH |   FILESIZE    TBR PROTO | VCODEC           VBR ACODEC      ABR ASR MORE INFO
-----------------------------------------------------------------------------------------------------------------------
sb2 mhtml 48x27        0    |                   mhtml | images                                   storyboard
sb1 mhtml 80x45        1    |                   mhtml | images                                   storyboard
sb0 mhtml 160x90       1    |                   mhtml | images                                   storyboard
139 m4a   audio only      2 |    1.24MiB    49k https | audio only           mp4a.40.5   49k 22k [en] low, m4a_dash
249 webm  audio only      2 |    1.17MiB    46k 

In [ ]:
print("hi")

hi


In [ ]:
!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best" --merge-output-format mp4 -o /tmp/test.mp4 https://www.youtube.com/watch?v=-AawZPTw8pY

[youtube] Extracting URL: https://www.youtube.com/watch?v=-AawZPTw8pY
[youtube] -AawZPTw8pY: Downloading webpage
[youtube] -AawZPTw8pY: Downloading android vr player API JSON
ERROR: [youtube] -AawZPTw8pY: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive"))

True


In [ ]:
print(load_checkpoint())

{'index': 1975, 'segments': 700}


In [ ]:
print(len(os.listdir(OUTPUT_DIR)))

709


In [ ]:
!yt-dlp -f mp4 https://www.youtube.com/watch?v=dQw4w9WgXcQ 2>&1 | head -20

         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in an mp4 container, use "-t mp4" instead.
         If you know what you are doing and want a pre-merged mp4 format, use "-f b[ext=mp4]" instead to suppress this warning
[youtube] Extracting URL: https://www.youtube.com/watch?v=dQw4w9WgXcQ
[youtube] dQw4w9WgXcQ: Downloading webpage
[youtube] dQw4w9WgXcQ: Downloading android vr player API JSON
[info] dQw4w9WgXcQ: Downloading 1 format(s): 18
[download] Destination: Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster) [dQw4w9WgXcQ].mp4
[download] 100% of   11.28MiB in 00:00:00 at 38.06MiB/s  


In [ ]:
import os

# Check if the BASE_DIR exists
if os.path.exists(BASE_DIR):
    print(f"Contents of {BASE_DIR}:")
    for item in os.listdir(BASE_DIR):
        print(item)
else:
    print(f"The directory {BASE_DIR} does not exist.")

Contents of /content/workspace/Drive/MyDrive/imagination_dataset:
segments
videos


In [ ]:
import os

print(f"Does {ANNOTATION_PATH} exist? {os.path.exists(ANNOTATION_PATH)}")

Does /content/workspace/Drive/MyDrive/imagination_dataset/VASTannotations150k.json exist? False


In [ ]:
print(f"Listing contents of {BASE_DIR}:")
!ls -l "{BASE_DIR}"

Listing contents of /content/workspace/Drive/MyDrive/imagination_dataset:
total 8
drwxr-xr-x 2 root root 4096 Apr  6 13:49 segments
drwxr-xr-x 2 root root 4096 Apr  6 13:49 videos


In [ ]:
!ls /content/drive/MyDrive/imagination_dataset/segments

seg_004707  seg_004733	seg_004759  seg_004785	seg_004811  seg_004837
seg_004708  seg_004734	seg_004760  seg_004786	seg_004812  seg_004838
seg_004709  seg_004735	seg_004761  seg_004787	seg_004813  seg_004839
seg_004710  seg_004736	seg_004762  seg_004788	seg_004814  seg_004840
seg_004711  seg_004737	seg_004763  seg_004789	seg_004815  seg_004841
seg_004712  seg_004738	seg_004764  seg_004790	seg_004816  seg_004842
seg_004713  seg_004739	seg_004765  seg_004791	seg_004817  seg_004843
seg_004714  seg_004740	seg_004766  seg_004792	seg_004818  seg_004844
seg_004715  seg_004741	seg_004767  seg_004793	seg_004819  seg_004845
seg_004716  seg_004742	seg_004768  seg_004794	seg_004820  seg_004846
seg_004717  seg_004743	seg_004769  seg_004795	seg_004821  seg_004847
seg_004718  seg_004744	seg_004770  seg_004796	seg_004822  seg_004848
seg_004719  seg_004745	seg_004771  seg_004797	seg_004823  seg_004849
seg_004720  seg_004746	seg_004772  seg_004798	seg_004824  seg_004850
seg_004721  seg_004747	seg_004773 

In [ ]:
!ls /content/drive/MyDrive/imagination_dataset/segments | wc -l

152


In [ ]:
!find /content/drive/MyDrive -type d -name "segments"

/content/drive/MyDrive/imagination_dataset/segments


In [ ]:
!ls /content/drive/MyDrive/imagination_dataset

checkpoint_v6.json  segments  videos


In [ ]:
!find /content/drive/MyDrive/imagination_dataset -type f -name "seg_*" | sort | head

In [ ]:
!find /content/drive/MyDrive/imagination_dataset -type f -name "seg_*" | sort | tail

In [ ]:
!find /content -type d -name segments

/content/drive/MyDrive/imagination_dataset/segments


In [ ]:
!find /content -type f -name "seg_*" | head -n 20

In [ ]:
import os

segments = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("seg_")]
nums = sorted([int(d.split("_")[1]) for d in segments])

print("Total:", len(nums))
print("Min:", nums[0])
print("Max:", nums[-1])

Total: 152
Min: 4707
Max: 4858


In [ ]:
print (OUTPUT_DIR)

/content/drive/MyDrive/imagination_dataset/segments


In [ ]:
import shutil

src = "/content/drive/MyDrive/imagination_dataset/segments"

zip_path = shutil.make_archive(
    "/content/segments_4707_plus",
    'zip',
    src
)

print("Created:", zip_path)

Created: /content/segments_4707_plus.zip


In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/imagination_dataset/checkpoint_v6.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

path = "/content/drive/MyDrive/imagination_dataset/segments"

segments = []
with os.scandir(path) as it:
    for entry in it:
        if entry.is_dir() and entry.name.startswith("seg_"):
            segments.append(entry.name)

print(len(segments))


152


In [ ]:
import os

path = "/content/drive/MyDrive/imagination_dataset/segments"

segments = sorted([
    d for d in os.listdir(path)
    if d.startswith("seg_")
])

print("Count:", len(segments))
print("First:", segments[:5])
print("Last:", segments[-5:])

Count: 152
First: ['seg_004707', 'seg_004708', 'seg_004709', 'seg_004710', 'seg_004711']
Last: ['seg_004854', 'seg_004855', 'seg_004856', 'seg_004857', 'seg_004858']


In [ ]:
import os

path = "/content/drive/MyDrive/imagination_dataset/segments"

segments = []
with os.scandir(path) as it:
    for entry in it:
        if entry.is_dir() and entry.name.startswith("seg_"):
            segments.append(entry.name)

print(len(segments))

152


In [ ]:
BASE_DIR        = "/content/drive/MyDrive/imagination_dataset"
TMP_VIDEO_DIR   = os.path.join(BASE_DIR, "videos")
OUTPUT_DIR      = os.path.join(BASE_DIR, "segments")   # one subfolder per segment
CHECKPOINT_PATH = os.path.join(BASE_DIR, "checkpoint_v6.json")
ANNOTATION_PATH = os.path.join(BASE_DIR, "VASTannotations150k.json")


In [ ]:
import os

segments = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("seg_")]
nums = sorted([int(d.split("_")[1]) for d in segments])

print("Total:", len(nums))
print("Min:", nums[0])
print("Max:", nums[-1])

Total: 5463
Min: 0
Max: 5462


In [ ]:
!ls /content/drive/MyDrive/imagination_dataset/segments | wc -l

5463
